In [7]:
import pandas as pd
import numpy as np


mobility = pd.read_csv("data/va_mobility.csv")
mobility['date'] = pd.to_datetime(mobility['date'])

na_ratio = mobility.isna().mean()
print(na_ratio)

features = [
    'retail_and_recreation_percent_change_from_baseline', 
    'grocery_and_pharmacy_percent_change_from_baseline', 
    'parks_percent_change_from_baseline', 
    'transit_stations_percent_change_from_baseline', 
    'workplaces_percent_change_from_baseline', 
    'residential_percent_change_from_baseline'
]

rural_counties = ["Accomack County","Alleghany County","Bath County",
                  "Bland County","Brunswick County","Buchanan County",
                  "Carroll County","Charlotte County","Craig County",
                  "Dickenson County","Essex County","Grayson County",
                  "Greensville County","Halifax County","Henry County",
                  "Highland County","Lee County","Louisa County",
                  "Lunenburg County","Madison County","Mecklenburg County",
                  "Middlesex County","Montgomery County","Nelson County",
                  "Northampton County","Northumberland County","Patrick County",
                  "Pittsylvania County","Prince Edward County","Pulaski County",
                  "Richmond County","Rockbridge County","Rockingham County",
                  "Russell County","Smyth County","Southampton County",
                  "Tazewell County","Wise County","Wythe County","Shenandoah County"]

mobility['metro_label'] = np.where(mobility['sub_region_2'].isin(rural_counties), 0, 1)

country_region_code                                   0.000000
country_region                                        0.000000
sub_region_1                                          0.000000
sub_region_2                                          0.012342
metro_area                                            1.000000
iso_3166_2_code                                       0.987658
census_fips_code                                      0.012342
place_id                                              0.000000
date                                                  0.000000
retail_and_recreation_percent_change_from_baseline    0.408797
grocery_and_pharmacy_percent_change_from_baseline     0.434545
parks_percent_change_from_baseline                    0.856850
transit_stations_percent_change_from_baseline         0.689028
workplaces_percent_change_from_baseline               0.026008
residential_percent_change_from_baseline              0.491897
dtype: float64


In [8]:
features_keep = [f for f in features if mobility[f].notna().mean() > 0.3]

print("Keeping features:", features_keep)

Keeping features: ['retail_and_recreation_percent_change_from_baseline', 'grocery_and_pharmacy_percent_change_from_baseline', 'transit_stations_percent_change_from_baseline', 'workplaces_percent_change_from_baseline', 'residential_percent_change_from_baseline']


In [9]:
mobility_filled = (
    mobility
    .sort_values(['sub_region_2', 'date'])
    .groupby('sub_region_2', group_keys=False)
    .apply(lambda df: df[features_keep].interpolate(limit_direction='both')
                          .ffill()
                          .bfill()
                          .assign(metro_label=df['metro_label'], date=df['date'], sub_region_2=df['sub_region_2']))
)

/var/folders/lv/9m_qv53944x04_1mpk21jgtc0000gn/T/ipykernel_79740/1547145550.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: df[features_keep].interpolate(limit_direction='both')


In [10]:
county_max_missing = mobility_filled.groupby('sub_region_2')[features_keep].apply(lambda df: df.isna().mean().max())
counties_keep = county_max_missing[county_max_missing < 0.7].index
mobility_clean = mobility_filled[mobility_filled['sub_region_2'].isin(counties_keep)]

In [11]:

print(len(mobility_clean['sub_region_2'].unique()))
print(mobility_clean[features_keep].isna().sum())

print(mobility_clean['metro_label'].sum()/len(mobility_clean))

mobility_clean.to_csv("data/mobility_clean.csv", index=False)

53
retail_and_recreation_percent_change_from_baseline    0
grocery_and_pharmacy_percent_change_from_baseline     0
transit_stations_percent_change_from_baseline         0
workplaces_percent_change_from_baseline               0
residential_percent_change_from_baseline              0
dtype: int64
0.8504166991198692


In [13]:
print(len(mobility.columns))
print(mobility_clean.size)

16
410848
